<a href="https://colab.research.google.com/github/tbcreator71/Google-Colab-Notebooks/blob/ComfyUI_Notebooks/LTX25_A100_80GB.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Notebook Version: v1.2.0-A100

**Updated:** 2026-11-11  |  **Profile:** Colab A100 80GB + LTX 2.5 + Drive persistent cache

> Verify this version at the top before running. Every future edit should increment the version number.

---

# ComfyUI + LTX 2.5 | A100 80GB BF16 (persistent cache)

**Recommended for repeated sessions:** Keep ComfyUI **and all model inference on Colab local disk**, and keep a **persistent backup on Google Drive**. The VM disk is ephemeral; Drive persists. On a new VM, the notebook restores models to local disk instead of running them directly from Drive.

Google Colab itself warns that Drive mount has high overhead for lots of small files, so the ComfyUI code + custom nodes are saved as **one compressed setup archive**. Large model weights are copied from Drive as **individual large files**, avoiding the overhead of repeatedly downloading them from Hugging Face. Python dependencies are installed on the fresh runtime rather than trying to transplant an incompatible preinstalled environment. See: https://research.google.com/colaboratory/faq.html.

**Use in order:** 1. Settings → 2. Bootstrap & restore → 3. Models & workflows → 4. Launch. Cell 5 is diagnostics; cell 6 saves outputs. On the first run, downloads and archive creation take longer. Later sessions restore from the archive, but **still need to transfer the model bytes** to the fast local VM storage; persistent Drive does not make this instantaneous.

**Safety:** The Cloudflare Quick Tunnel URL has no access control. Keep it private and stop Colab when finished. Set optional Hugging Face gated-access token only in Colab Secrets (`HF_TOKEN`), never inside the notebook.

In [ ]:
!nvidia-smi

In [ ]:
#@title 1. Settings
from pathlib import Path
WORKSPACE = Path("/content/ComfyUI")  # Always use Colab's FAST local VM disk for all inference
USE_DRIVE_MODEL_ARCHIVE = True  #@param {type:"boolean"}
ARCHIVE_ROOT = "/content/drive/MyDrive/LTX25-Colab-Model-Archive"  #@param {type:"string"}
RESTORE_SETUP_ARCHIVE = True  #@param {type:"boolean"}
# Restore ComfyUI and custom nodes as one compressed archive, avoiding slow small-file Drive I/O.
REBUILD_SETUP_ARCHIVE = False  #@param {type:"boolean"}
# Set True once when you want to reinstall latest ComfyUI + nodes and refresh the setup archive.
SAVE_OUTPUTS_TO_DRIVE = True  #@param {type:"boolean"}
INSTALL_PROMPT_ENHANCER = True  #@param {type:"boolean"}
print("Model persistence:", USE_DRIVE_MODEL_ARCHIVE, "| Setup archive:", RESTORE_SETUP_ARCHIVE)
print("Local disk:", WORKSPACE, "| Drive archive:", ARCHIVE_ROOT)

In [ ]:
#@title 2. Restore or install ComfyUI + LTX nodes (local SSD)
import os, sys, subprocess, shutil, tarfile, time
from pathlib import Path
def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, check=True)

ARCHIVE = None
if USE_DRIVE_MODEL_ARCHIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARCHIVE = Path(ARCHIVE_ROOT)
    ARCHIVE.mkdir(parents=True, exist_ok=True)

SETUP_TAR = ARCHIVE / "comfyui-setup-v1.tar.gz" if ARCHIVE else None
if REBUILD_SETUP_ARCHIVE:
    shutil.rmtree(WORKSPACE, ignore_errors=True)

if not (WORKSPACE / "main.py").exists() and RESTORE_SETUP_ARCHIVE and SETUP_TAR and SETUP_TAR.is_file() and not REBUILD_SETUP_ARCHIVE:
    print("Restoring small-file ComfyUI setup from one archive on Drive...", flush=True)
    with tarfile.open(SETUP_TAR, "r:gz") as tar:
        # This archive is created by this notebook. Do not use untrusted tar files.
        tar.extractall(path="/content", filter="data")
    print("Setup restored:", WORKSPACE)

def clone_or_pull(url, directory):
    directory = Path(directory)
    if (directory / ".git").exists():
        # Existing installations are reused to avoid breaking a working ComfyUI setup.
        return
    if not directory.exists():
        run("git", "clone", "--depth", "1", url, directory)

run("nvidia-smi")
free_gib = shutil.disk_usage("/content").free / 2**30
print("Local free space:", round(free_gib, 1), "GiB")
if free_gib < 90:
    print("WARNING: full BF16 model set may exceed local disk capacity. Check Colab's VM disk.")
run("apt-get", "update", "-qq")
run("apt-get", "install", "-y", "-qq", "ffmpeg")
run(sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "requests", "safetensors")
clone_or_pull("https://github.com/comfyanonymous/ComfyUI.git", WORKSPACE)
run(sys.executable, "-m", "pip", "install", "-q", "-r", WORKSPACE / "requirements.txt")
custom = WORKSPACE / "custom_nodes"
clone_or_pull("https://github.com/ltdrdata/ComfyUI-Manager.git", custom / "ComfyUI-Manager")
clone_or_pull("https://github.com/Lightricks/ComfyUI-LTXVideo.git", custom / "ComfyUI-LTXVideo")
ltx_req = custom / "ComfyUI-LTXVideo" / "requirements.txt"
if ltx_req.exists():
    run(sys.executable, "-m", "pip", "install", "-q", "-r", ltx_req)
for folder in ("diffusion_models", "text_encoders", "vae", "latent_upscale_models", "loras"):
    (WORKSPACE / "models" / folder).mkdir(parents=True, exist_ok=True)
(WORKSPACE / "user" / "default" / "workflows").mkdir(parents=True, exist_ok=True)
run(sys.executable, "-c", "import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))")

if SETUP_TAR and (REBUILD_SETUP_ARCHIVE or not SETUP_TAR.exists()):
    print("Saving one small-file setup archive to Drive for future sessions...", flush=True)
    temp_tar = SETUP_TAR.with_suffix(".tar.gz.partial")
    with tarfile.open(temp_tar, "w:gz") as tar:
        # Save code + custom nodes only, NOT large model weights, outputs or generated media.
        for root, dirs, files in os.walk(WORKSPACE):
            here = Path(root)
            dirs[:] = [d for d in dirs if d not in (".git","__pycache__",".cache","models","output","input","temp")]
            for fn in files:
                if fn.endswith((".pyc", ".safetensors", ".ckpt", ".pt")): continue
                file = here / fn
                tar.add(file, arcname=file.relative_to(Path("/content")), recursive=False)
    os.replace(temp_tar, SETUP_TAR)
    print("Saved:", SETUP_TAR)
print("Ready. ComfyUI operates entirely from", WORKSPACE)

In [ ]:
#@title 3. Download required LTX 2.5 BF16 models + official ready-to-open workflows
import os, json, shutil, requests, time, concurrent.futures
from pathlib import Path
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

# Compatible with official 2.5 two-stage, single-stage and audio-to-video graphs.
# Files from the same official HF subfolders are placed directly under ComfyUI/models.
MODEL_FILES = [
    ("Lightricks/LTX-2.5", "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-video-vae-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"),
]
if INSTALL_PROMPT_ENHANCER:
    MODEL_FILES.append(("Comfy-Org/gemma-4", "gemma4_e2b_it_bf16.safetensors"))

# Persistent Drive archive -> copy big weights onto FAST local disk; never infer from Drive.
archive = ARCHIVE if USE_DRIVE_MODEL_ARCHIVE else None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None

# Check model authorization BEFORE any expensive multi-GB downloads.
from huggingface_hub import HfApi
if not HF_TOKEN:
    raise RuntimeError("Missing HF_TOKEN. In Colab: Secrets (key icon) → add HF_TOKEN → enable notebook access. First accept the license at https://huggingface.co/Lightricks/LTX-2.5")
try:
    HfApi().model_info("Lightricks/LTX-2.5", token=HF_TOKEN)
    # model_info may be accessible without weight access, so test metadata for the actual gated file.
    from huggingface_hub import get_hf_file_metadata, hf_hub_url
    get_hf_file_metadata(hf_hub_url("Lightricks/LTX-2.5", "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors"), token=HF_TOKEN)
except Exception as exc:
    raise RuntimeError("LTX 2.5 gated model access is not authorized. Accept its license at huggingface.co/Lightricks/LTX-2.5 and provide a Read token in Colab Secrets (HF_TOKEN).") from exc
print("Hugging Face model access verified.")

def file_ok(p, minimum=100_000_000):
    return p.is_file() and p.stat().st_size > minimum

# Parallelize only 2 large weight transfers. Drive throttling makes high concurrency unreliable.
def install_weight(entry):
    repo_id, rel = entry
    relpath = Path(rel if repo_id != "Comfy-Org/gemma-4" else "text_encoders/" + Path(rel).name)
    target = WORKSPACE / "models" / relpath
    stored = archive / relpath if archive else None
    target.parent.mkdir(parents=True, exist_ok=True)
    if file_ok(target):
        return ("Already local", target.name)
    if stored and file_ok(stored):
        print("Restoring from Drive:", target.name, flush=True)
        tmp = target.with_suffix(".safetensors.partial")
        try:
            shutil.copyfile(stored, tmp)
            if tmp.stat().st_size != stored.stat().st_size:
                raise IOError("Incomplete Drive restore: " + target.name)
            os.replace(tmp, target)
        finally:
            if tmp.exists(): tmp.unlink()
        return ("Restored", target.name)
    print("Downloading from Hugging Face:", repo_id, relpath, flush=True)
    try:
        hf_hub_download(repo_id=repo_id, filename=relpath.as_posix(), local_dir=WORKSPACE / "models", token=HF_TOKEN)
    except HfHubHTTPError as exc:
        if getattr(exc.response, "status_code", None) in (401, 403):
            raise RuntimeError("Accept the LTX model license on Hugging Face; add HF_TOKEN in Colab Secrets.") from exc
        raise
    if not file_ok(target):
        raise RuntimeError("Downloaded model missing or unexpectedly small: " + str(target))
    if stored:
        stored.parent.mkdir(parents=True, exist_ok=True)
        if not file_ok(stored):
            print("Backing up to Drive:", target.name, flush=True)
            tmp = stored.with_suffix(".safetensors.partial")
            try:
                shutil.copyfile(target, tmp)
                if tmp.stat().st_size != target.stat().st_size:
                    raise IOError("Incomplete Drive archive: " + target.name)
                os.replace(tmp, stored)
            finally:
                if tmp.exists(): tmp.unlink()
    return ("Downloaded and archived" if stored else "Downloaded", target.name)

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(install_weight, MODEL_FILES))
for state, filename in results: print(state, ":", filename)
OFFICIAL = "https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.5/"
WORKFLOWS = [
    "LTX-2.5_T2V_I2V_Two_Stage_Distilled.json",
    "LTX-2.5_T2V_I2V_Single_Stage_Distilled.json",
    "LTX-2.5_A2V_Two_Stage_Distilled.json",
    "LTX-2.5_ICLoRA_Ingredients_Single_Stage_Distilled.json",
]
wf_dir = WORKSPACE / "user" / "default" / "workflows"
for filename in WORKFLOWS:
    response = requests.get(OFFICIAL + filename, timeout=120)
    response.raise_for_status()
    json.loads(response.text)
    (wf_dir / filename).write_text(response.text)
    print("Workflow ready:", filename)
print("\nAll essential model files are staged on FAST local disk.")
print("Workflows folder:", wf_dir)
print("Use two-stage T2V/I2V for quality; single-stage for previews; A2V for existing dialogue.")
print("Ingredients IC-LoRA workflow needs an EXTRA LoRA model. It is not predownloaded; use ComfyUI Missing Models when needed.")
print("Note: if using INSTALL_PROMPT_ENHANCER=False, turn off prompt enhancement in the workflow.")

In [ ]:
#@title 4. Start ComfyUI using ONE Cloudflare Quick Tunnel
import os, re, time, subprocess, threading, urllib.request
from pathlib import Path

# Only create a tunnel for the current VM; never expose this URL publicly.
if subprocess.run(["bash","-lc","command -v cloudflared"],capture_output=True).returncode:
    deb = Path("/content/cloudflared.deb")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", deb)
    subprocess.run(["dpkg","-i",str(deb)],check=True)

# Restart-safe within the SAME runtime, but do not run this cell in parallel.
for key in ("COMFY_PROCESS", "CLOUDFLARED_PROCESS"):
    process = globals().get(key)
    if process is not None and process.poll() is None:
        process.terminate()
        try: process.wait(timeout=8)
        except subprocess.TimeoutExpired: process.kill()
time.sleep(2)

log = open("/content/comfyui.log","a",buffering=1)
COMFY_PROCESS = subprocess.Popen(
    ["python","main.py","--listen","127.0.0.1","--port","8188"],
    cwd=str(WORKSPACE),stdout=log,stderr=subprocess.STDOUT)
print("ComfyUI PID:", COMFY_PROCESS.pid)

for _ in range(150):
    if COMFY_PROCESS.poll() is not None:
        raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log (see diagnostics cell).")
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats",timeout=2).read()
        break
    except Exception: time.sleep(2)
else:
    raise TimeoutError("ComfyUI did not become ready; inspect /content/comfyui.log")

tunnel_log = open("/content/cloudflared.log","a",buffering=1)
CLOUDFLARED_PROCESS = subprocess.Popen(
    ["cloudflared","tunnel","--no-autoupdate","--url","http://127.0.0.1:8188"],
    stdout=tunnel_log,stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    text = Path("/content/cloudflared.log").read_text(errors="replace")
    matches = re.findall(r"https://[\w-]+\.trycloudflare\.com", text)
    if matches: url = matches[-1]; break
    if CLOUDFLARED_PROCESS.poll() is not None: break
    time.sleep(2)
if not url:
    raise RuntimeError("Tunnel did not return a URL. Inspect /content/cloudflared.log.")
print("\nOPEN COMFYUI:", url)
print("Use Workflows → Open and select LTX-2.5_T2V_I2V_Two_Stage_Distilled.json")
print("Keep Colab runtime connected while generating. Stop runtime to stop billing.")
print("Cloudflare Quick Tunnel is not password protected. Keep this URL private.")

In [ ]:
#@title 5. Diagnostics (run only when something fails)
from pathlib import Path
import shutil, subprocess
print("Free local disk GiB:",round(shutil.disk_usage("/content").free / 2**30,1))
subprocess.run(["nvidia-smi"])
for name in ("comfyui.log","cloudflared.log"):
    p=Path("/content")/name
    if p.exists():
        print("\n---",name,"(last 70 lines) ---")
        print("\n".join(p.read_text(errors="replace").splitlines()[-70:]))

In [ ]:
#@title 6. Optional: back up finished videos to Google Drive (not used during inference)
from pathlib import Path
import shutil
if not SAVE_OUTPUTS_TO_DRIVE:
    print("Output backup disabled. Videos are in", WORKSPACE / "output")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    src = WORKSPACE / "output"
    dst = Path("/content/drive/MyDrive/LTX25-Colab-Outputs")
    dst.mkdir(parents=True,exist_ok=True)
    copied=0
    for f in src.rglob("*"):
        if f.is_file():
            target=dst / f.relative_to(src)
            target.parent.mkdir(parents=True,exist_ok=True)
            if not target.exists() or f.stat().st_size!=target.stat().st_size:
                shutil.copy2(f,target)
                copied+=1
    print("New/changed files backed up:", copied, "Destination:",dst)